# Online Dataset Collector

Downloads and processes two online engagement datasets, extracts facial landmark features per image, and saves to `labeled_features.csv`.

**Works on:** Google Colab and locally in VSCode / Jupyter

**Datasets:**
- [Kaggle — Drowsy Dataset](https://www.kaggle.com/datasets/shivampandey1233/drowsy-dataset)
- [Roboflow — User Attention](https://universe.roboflow.com/neurosense/user-attention)

**Output columns:** `ear_left, ear_right, ear_avg, mar, pitch, yaw, roll, perclos, blink_rate, label`

Labels: `0 = ATTENTIVE` | `1 = SLEEPY` | `2 = DISTRACTED`

> `perclos` and `blink_rate` will be `0.0` for all rows — these require video, not static images.

The two datasets links:
* https://www.kaggle.com/datasets/shivampandey1233/drowsy-dataset
* https://universe.roboflow.com/neurosense/user-attention
* https://universe.roboflow.com/distractless/distractless

Potential Test Dataset:
* https://universe.roboflow.com/bklab/students-in-lecture
* https://universe.roboflow.com/123-cpztz/ml-pjutg

## Step 1: Install Dependencies

In [1]:
!pip install "numpy<2.0"
!pip install mediapipe opencv-python roboflow kaggle matplotlib pandas tqdm

  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl (12.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.17.0 requires numpy<2.0.0,>=1.23.5; python_version <= "3.11", but you have numpy 2.4.6 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Detect Environment + Mount Drive

Automatically detects whether running on Colab or locally.

In [ ]:
s

Running locally.


## Step 3: Configuration

Fill in your API credentials below:

| Key | Where to get it |
|---|---|
| `KAGGLE_USERNAME` | kaggle.com → Profile → Account → Username |
| `KAGGLE_KEY` | kaggle.com → Settings → API → **Create New Token** → copy the `key` value from the downloaded JSON |
| `ROBOFLOW_API_KEY` | roboflow.com → Account → Roboflow API → Private API Key |

Leave `KAGGLE_USERNAME` / `KAGGLE_KEY` empty to fall back to `~/.kaggle/kaggle.json` instead.

In [ ]:
from pathlib import Path

# --- Kaggle credentials (paste here instead of using kaggle.json) ---
KAGGLE_USERNAME  = ""   # e.g. "johndoe"
KAGGLE_KEY       = ""   # the "key" field from kaggle.json

# --- Roboflow ---
ROBOFLOW_API_KEY = ""

if ON_COLAB:
    OUTPUT_CSV   = Path('/content/drive/MyDrive/COS40007_Project/landmark_pipeline/labeled_features.csv')
    KAGGLE_DIR   = Path('/content/kaggle_drowsy')
    ROBOFLOW_DIR = Path('/content/roboflow_attention')
else:
    _HERE        = Path('__file__').parent if '__file__' in dir() else Path('.')
    _ROOT        = _HERE.parent.parent.parent.parent
    OUTPUT_CSV   = _ROOT / 'data' / 'labeled_features.csv'
    KAGGLE_DIR   = _ROOT / 'data' / 'kaggle_drowsy'
    ROBOFLOW_DIR = _ROOT / 'data' / 'roboflow_attention'

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
ROBOFLOW_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ['ear_left', 'ear_right', 'ear_avg', 'mar',
                'pitch', 'yaw', 'roll', 'perclos', 'blink_rate']
LABEL_NAMES  = {0: 'ATTENTIVE', 1: 'SLEEPY', 2: 'DISTRACTED'}

KAGGLE_LABEL_MAP = {
    'drowsy':     1,
    'sleepy':     1,
    'yawn':       1,
    'non_drowsy': 0,
    'awake':      0,
    'alert':      0,
    'nodding':    0,
}
ROBOFLOW_LABEL_MAP = {
    'attentive':   0,
    'focused':     0,
    'engaged':     0,
    'drowsy':      1,
    'sleepy':      1,
    'distracted':  2,
    'inattentive': 2,
}

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print(f'Output CSV   : {OUTPUT_CSV}')
print(f'Kaggle dir   : {KAGGLE_DIR}')
print(f'Roboflow dir : {ROBOFLOW_DIR}')

## Step 4: Download Kaggle Dataset

**Colab:** upload `kaggle.json` when prompted.

**Local:** credentials are read from `KAGGLE_USERNAME` / `KAGGLE_KEY` set in Step 3.  
Falls back to `~/.kaggle/kaggle.json` if both fields are left empty.

In [10]:
import os

if ON_COLAB:
    from google.colab import files
    print('Upload kaggle.json:')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
else:
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
        os.environ['KAGGLE_KEY']      = KAGGLE_KEY
        print('Kaggle credentials set from notebook.')
    elif (Path.home() / '.kaggle' / 'kaggle.json').exists():
        print(f'Using kaggle.json at {Path.home() / ".kaggle" / "kaggle.json"}')
    else:
        print('No Kaggle credentials found.')
        print('Fill in KAGGLE_USERNAME and KAGGLE_KEY in Step 3, or place kaggle.json at:')
        print(f'  {Path.home() / ".kaggle" / "kaggle.json"}')
        raise RuntimeError('Kaggle credentials missing')

get_ipython().system(f'kaggle datasets download -d shivampandey1233/drowsy-dataset -p "{KAGGLE_DIR}" --unzip')

print('\nDownloaded folders:')
for item in KAGGLE_DIR.rglob('*'):
    if item.is_dir():
        count = len(list(item.glob('*.*')))
        if count > 0:
            print(f'  {item.name}/  ({count} files)')

Kaggle credentials set from notebook.
Dataset URL: https://www.kaggle.com/datasets/shivampandey1233/drowsy-dataset
License(s): apache-2.0


Downloaded folders:
  Active Subjects/  (787 files)
  Fatigue Subjects/  (799 files)
  Yawning Subjects/  (298 files)



  0%|          | 0.00/79.9M [00:00<?, ?B/s]
  1%|▏         | 1.00M/79.9M [00:01<01:33, 881kB/s]
  3%|▎         | 2.00M/79.9M [00:01<00:45, 1.78MB/s]
  4%|▍         | 3.00M/79.9M [00:01<00:28, 2.86MB/s]
  5%|▌         | 4.00M/79.9M [00:01<00:20, 3.95MB/s]
  8%|▊         | 6.00M/79.9M [00:01<00:11, 6.69MB/s]
 11%|█▏        | 9.00M/79.9M [00:01<00:06, 10.9MB/s]
 15%|█▌        | 12.0M/79.9M [00:01<00:04, 14.5MB/s]
 19%|█▉        | 15.0M/79.9M [00:02<00:03, 17.3MB/s]
 21%|██▏       | 17.0M/79.9M [00:02<00:03, 17.7MB/s]
 24%|██▍       | 19.0M/79.9M [00:02<00:03, 16.7MB/s]
 28%|██▊       | 22.0M/79.9M [00:02<00:03, 18.4MB/s]
 30%|███       | 24.0M/79.9M [00:02<00:03, 18.6MB/s]
 34%|███▍      | 27.0M/79.9M [00:02<00:02, 20.3MB/s]
 38%|███▊      | 30.0M/79.9M [00:02<00:03, 15.7MB/s]
 41%|████▏     | 33.0M/79.9M [00:03<00:02, 17.8MB/s]
 44%|████▍     | 35.0M/79.9M [00:03<00:02, 17.5MB/s]
 48%|████▊     | 38.0M/79.9M [00:03<00:02, 20.5MB/s]
 51%|█████▏    | 41.0M/79.9M [00:03<00:01, 22.1MB/s]
 5

## Step 5: Download Roboflow Dataset

In [11]:
from roboflow import Roboflow
import shutil

# Remove empty/stale folder so Roboflow doesn't skip the download
if ROBOFLOW_DIR.exists() and not any(ROBOFLOW_DIR.rglob('*.jpg')) and not any(ROBOFLOW_DIR.rglob('*.png')):
    shutil.rmtree(ROBOFLOW_DIR)
    print(f'Removed empty folder: {ROBOFLOW_DIR}')

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('neurosense').project('user-attention')
dataset = project.version(1).download('yolov8', location=str(ROBOFLOW_DIR), overwrite=True)

print('\nDownloaded structure:')
for item in sorted(ROBOFLOW_DIR.rglob('*'))[:30]:
    print(f'  {item.relative_to(ROBOFLOW_DIR)}')

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data\roboflow_attention in yolov8:: 100%|██████████| 2862/2862 [00:00<00:00, 3631.95it/s]



Downloaded structure:
  data.yaml
  README.dataset.txt
  README.roboflow.txt
  test
  test\images
  test\images\Distracted-2_mp4-42_jpg.rf.78340d4d6fba5c2fe38c08dddee1f597.jpg
  test\images\Distracted-2_mp4-43_jpg.rf.a4bfacaa0191c7657dfca087b2f5d2d5.jpg
  test\images\Distracted-2_mp4-46_jpg.rf.67e35702fb2e8cf5b5cdad2c24e6ec40.jpg
  test\images\Distracted-2_mp4-54_jpg.rf.560a0ca65db354a0cd5506a0cd00fcd6.jpg
  test\images\Distracted-2_mp4-82_jpg.rf.194a1f2a129df7a6da6e6763d7a21216.jpg
  test\images\Distracted-2_mp4-84_jpg.rf.9e4e6e5e09640df0cd2f69d59eaedfce.jpg
  test\images\Distracted_mp4-104_jpg.rf.4c5aa21fc0b6d7b04a206d11aa81febf.jpg
  test\images\Distracted_mp4-107_jpg.rf.1014a5512f0a097fa9d69b88e8deb147.jpg
  test\images\Distracted_mp4-108_jpg.rf.e3e56b04458e28ca13d0a534a6a9be35.jpg
  test\images\Distracted_mp4-12_jpg.rf.308ac9529bbdceb162879e861ad54da4.jpg
  test\images\Distracted_mp4-44_jpg.rf.834191177ebe6a8394565c4bd772e97c.jpg
  test\images\Distracted_mp4-45_jpg.rf.b4da3f72bcc

## Step 6: Feature Extraction Setup

In [12]:
import cv2
import numpy as np
import mediapipe as mp

RIGHT_EYE    = [33, 160, 158, 133, 153, 144]
LEFT_EYE     = [362, 385, 387, 263, 373, 380]
MOUTH        = [61, 82, 312, 291, 317, 87]
HEAD_POSE_LM = [1, 152, 263, 33, 287, 57]
KEY_INDICES  = set(RIGHT_EYE + LEFT_EYE + MOUTH)

_MODEL_3D = np.array([
    [  0.0,    0.0,    0.0],
    [  0.0, -330.0,  -65.0],
    [-225.0,  170.0, -135.0],
    [ 225.0,  170.0, -135.0],
    [-150.0, -150.0, -125.0],
    [ 150.0, -150.0, -125.0],
], dtype=np.float64)


def _aspect_ratio(lm, indices):
    pts = np.array([[lm[i][0], lm[i][1]] for i in indices])
    v1  = np.linalg.norm(pts[1] - pts[5])
    v2  = np.linalg.norm(pts[2] - pts[4])
    h   = np.linalg.norm(pts[0] - pts[3])
    return float((v1 + v2) / (2.0 * h)) if h > 1e-6 else 0.0


def _head_pose(lm, cam_matrix):
    dist    = np.zeros((4, 1))
    img_pts = np.array([[lm[i][0], lm[i][1]] for i in HEAD_POSE_LM], dtype=np.float64)
    ok, rvec, _ = cv2.solvePnP(_MODEL_3D, img_pts, cam_matrix, dist,
                                flags=cv2.SOLVEPNP_ITERATIVE)
    if not ok:
        return 0.0, 0.0, 0.0
    rmat, _ = cv2.Rodrigues(rvec)
    sy = np.sqrt(rmat[0, 0] ** 2 + rmat[1, 0] ** 2)
    if sy > 1e-6:
        pitch = float(np.degrees(np.arctan2( rmat[2, 1], rmat[2, 2])))
        yaw   = float(np.degrees(np.arctan2(-rmat[2, 0], sy)))
        roll  = float(np.degrees(np.arctan2( rmat[1, 0], rmat[0, 0])))
    else:
        pitch = float(np.degrees(np.arctan2(-rmat[1, 2], rmat[1, 1])))
        yaw   = float(np.degrees(np.arctan2(-rmat[2, 0], sy)))
        roll  = 0.0
    return pitch, yaw, roll


def extract_features_and_bbox(img_bgr):
    h, w    = img_bgr.shape[:2]
    cam_mat = np.array([[w, 0, w/2],[0, w, h/2],[0, 0, 1]], dtype=np.float64)
    with mp.solutions.face_mesh.FaceMesh(
            static_image_mode=True, max_num_faces=1,
            min_detection_confidence=0.5) as fm:
        result = fm.process(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    if not result.multi_face_landmarks:
        return None, None, img_bgr
    raw = result.multi_face_landmarks[0].landmark
    lm  = [(int(p.x * w), int(p.y * h), p.z) for p in raw]
    xs  = [p[0] for p in lm]
    ys  = [p[1] for p in lm]
    pad = int((max(xs) - min(xs)) * 0.12)
    x1  = max(0,     min(xs) - pad)
    y1  = max(0,     min(ys) - pad)
    x2  = min(w - 1, max(xs) + pad)
    y2  = min(h - 1, max(ys) + pad)
    ear_l = _aspect_ratio(lm, LEFT_EYE)
    ear_r = _aspect_ratio(lm, RIGHT_EYE)
    mar   = _aspect_ratio(lm, MOUTH)
    pitch, yaw, roll = _head_pose(lm, cam_mat)
    features = {
        'ear_left':   round(ear_l,           4),
        'ear_right':  round(ear_r,           4),
        'ear_avg':    round((ear_l+ear_r)/2, 4),
        'mar':        round(mar,             4),
        'pitch':      round(pitch,           4),
        'yaw':        round(yaw,             4),
        'roll':       round(roll,            4),
        'perclos':    0.0,
        'blink_rate': 0.0,
    }
    annotated = img_bgr.copy()
    cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
    for i in KEY_INDICES:
        cv2.circle(annotated, (lm[i][0], lm[i][1]), 2, (0, 230, 230), -1)
    return features, (x1, y1, x2, y2), annotated


print('Feature extraction functions loaded.')

NameError: name 'core' is not defined

## Step 7: Visualise Sample Detections

Shows bounding boxes and landmark dots on the first images from each dataset.

In [ ]:
import matplotlib.pyplot as plt
import yaml


# --- Kaggle: subdirectory names are labels ---
def show_samples(dataset_dir, label_map, title, n=12):
    samples = []
    for d in sorted(Path(dataset_dir).rglob('*')):
        if not d.is_dir() or d.name.lower() not in label_map:
            continue
        for p in list(d.glob('*.*'))[:4]:
            if p.suffix.lower() in IMAGE_EXTENSIONS:
                samples.append((p, d.name.lower()))
        if len(samples) >= n:
            break
    if not samples:
        print('No matching folders. Available:', [d.name for d in Path(dataset_dir).rglob('*') if d.is_dir()])
        return
    cols = 4
    rows = (min(len(samples), n) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    axes = np.array(axes).flatten()
    found = 0
    for idx, (img_path, key) in enumerate(samples[:n]):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        features, bbox, annotated = extract_features_and_bbox(img)
        axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[idx].axis('off')
        label_name = LABEL_NAMES[label_map[key]]
        status     = 'detected' if features else 'no face'
        axes[idx].set_title(f'{label_name}  [{status}]', fontsize=9)
        if features:
            found += 1
    for ax in axes[len(samples):]:
        ax.axis('off')
    plt.suptitle(f'{title}  ({found}/{min(len(samples), n)} faces found)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# --- Roboflow: YOLO format (images/ + labels/ txt files) ---
def show_yolo_samples(dataset_dir, title, n=12):
    dataset_dir = Path(dataset_dir)
    yaml_files  = list(dataset_dir.rglob('data.yaml'))
    if not yaml_files:
        print(f'data.yaml not found in {dataset_dir}')
        return
    with open(yaml_files[0]) as f:
        cfg = yaml.safe_load(f)
    class_names = cfg.get('names', [])

    samples = []
    for split in ['train', 'valid', 'test']:
        images_dir = dataset_dir / split / 'images'
        labels_dir = dataset_dir / split / 'labels'
        if not images_dir.exists():
            continue
        for img_path in sorted(images_dir.glob('*.*'))[:4]:
            if img_path.suffix.lower() in IMAGE_EXTENSIONS:
                lp = labels_dir / (img_path.stem + '.txt')
                samples.append((img_path, lp))
        if len(samples) >= n:
            break

    if not samples:
        print(f'No images found under {dataset_dir}/*/images/')
        return

    cols  = 4
    rows  = (min(len(samples), n) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    axes  = np.array(axes).flatten()
    found = 0
    for idx, (img_path, lp) in enumerate(samples[:n]):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w      = img.shape[:2]
        annotated = img.copy()
        lbls      = []
        if lp.exists():
            for line in lp.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                cid = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = max(0,     int((cx - bw / 2) * w))
                y1 = max(0,     int((cy - bh / 2) * h))
                x2 = min(w - 1, int((cx + bw / 2) * w))
                y2 = min(h - 1, int((cy + bh / 2) * h))
                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
                if cid < len(class_names):
                    lbls.append(class_names[cid])
        axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[idx].axis('off')
        axes[idx].set_title(', '.join(lbls) or 'no label', fontsize=9)
        found += 1
    for ax in axes[len(samples):]:
        ax.axis('off')
    plt.suptitle(f'{title}  ({found}/{min(len(samples), n)} shown)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


show_samples(KAGGLE_DIR, KAGGLE_LABEL_MAP, 'Kaggle — Drowsy Dataset')
show_yolo_samples(ROBOFLOW_DIR, 'Roboflow — User Attention (YOLO format)')

## Step 8: Process All Images

In [ ]:
import yaml
from tqdm.notebook import tqdm


# --- Kaggle: folder-per-class layout ---
def process_kaggle(dataset_dir, label_map, name):
    rows    = []
    counts  = {v: 0 for v in LABEL_NAMES.values()}
    skipped = 0
    dirs    = [d for d in sorted(Path(dataset_dir).rglob('*'))
               if d.is_dir() and d.name.lower() in label_map]
    if not dirs:
        print(f'[{name}] No matching folders. Check label map.')
        print('Available:', [d.name for d in Path(dataset_dir).rglob('*') if d.is_dir()])
        return rows
    for d in dirs:
        key    = d.name.lower()
        lbl_id = label_map[key]
        images = [p for p in d.glob('*.*') if p.suffix.lower() in IMAGE_EXTENSIONS]
        print(f'  {d.name}/  ->  {LABEL_NAMES[lbl_id]}  ({len(images)} images)')
        for img_path in tqdm(images, desc=d.name, leave=False):
            img = cv2.imread(str(img_path))
            if img is None:
                skipped += 1
                continue
            features, _, _ = extract_features_and_bbox(img)
            if features is None:
                skipped += 1
                continue
            row = {col: features[col] for col in FEATURE_COLS}
            row['label'] = lbl_id
            rows.append(row)
            counts[LABEL_NAMES[lbl_id]] += 1
    print(f'[{name}]', '  '.join(f"{k}: {v}" for k, v in counts.items()),
          f'  skipped: {skipped}')
    return rows


# --- Roboflow: YOLO format — crop each annotation bbox, then run MediaPipe ---
def process_roboflow_yolo(dataset_dir, name):
    dataset_dir = Path(dataset_dir)
    yaml_files  = list(dataset_dir.rglob('data.yaml'))
    if not yaml_files:
        print(f'[{name}] data.yaml not found. Cannot determine class names.')
        return []
    with open(yaml_files[0]) as f:
        cfg = yaml.safe_load(f)
    class_names = cfg.get('names', [])
    print(f'  Classes in data.yaml: {class_names}')

    id_to_label = {}
    for i, cls_name in enumerate(class_names):
        key = cls_name.lower()
        if key in ROBOFLOW_LABEL_MAP:
            id_to_label[i] = ROBOFLOW_LABEL_MAP[key]

    if not id_to_label:
        print(f'[{name}] No class names matched ROBOFLOW_LABEL_MAP.')
        print(f'  data.yaml classes : {class_names}')
        print(f'  ROBOFLOW_LABEL_MAP: {list(ROBOFLOW_LABEL_MAP.keys())}')
        return []

    rows    = []
    counts  = {v: 0 for v in LABEL_NAMES.values()}
    skipped = 0

    for split in ['train', 'valid', 'test']:
        images_dir = dataset_dir / split / 'images'
        labels_dir = dataset_dir / split / 'labels'
        if not images_dir.exists():
            continue
        images = [p for p in images_dir.glob('*.*') if p.suffix.lower() in IMAGE_EXTENSIONS]
        print(f'  {split}/  ({len(images)} images)')
        for img_path in tqdm(images, desc=split, leave=False):
            img = cv2.imread(str(img_path))
            if img is None:
                skipped += 1
                continue
            h, w  = img.shape[:2]
            lp    = labels_dir / (img_path.stem + '.txt')
            if not lp.exists():
                skipped += 1
                continue
            for line in lp.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                cid = int(parts[0])
                if cid not in id_to_label:
                    continue
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = max(0,     int((cx - bw / 2) * w))
                y1 = max(0,     int((cy - bh / 2) * h))
                x2 = min(w - 1, int((cx + bw / 2) * w))
                y2 = min(h - 1, int((cy + bh / 2) * h))
                crop = img[y1:y2, x1:x2]
                if crop.size == 0:
                    skipped += 1
                    continue
                features, _, _ = extract_features_and_bbox(crop)
                if features is None:
                    skipped += 1
                    continue
                row = {col: features[col] for col in FEATURE_COLS}
                row['label'] = id_to_label[cid]
                rows.append(row)
                counts[LABEL_NAMES[id_to_label[cid]]] += 1

    print(f'[{name}]', '  '.join(f"{k}: {v}" for k, v in counts.items()),
          f'  skipped: {skipped}')
    return rows


print('Processing Kaggle dataset...')
rows_kaggle   = process_kaggle(KAGGLE_DIR, KAGGLE_LABEL_MAP, 'Kaggle')
print('\nProcessing Roboflow dataset...')
rows_roboflow = process_roboflow_yolo(ROBOFLOW_DIR, 'Roboflow')

all_rows = rows_kaggle + rows_roboflow
print(f'\nTotal rows: {len(all_rows)}')

## Step 9: Save to labeled_features.csv (overwrite)

In [ ]:
import pandas as pd

if not all_rows:
    print('No rows collected. Check label maps match downloaded folder names.')
else:
    df = pd.DataFrame(all_rows, columns=FEATURE_COLS + ['label'])
    df.to_csv(OUTPUT_CSV, index=False)
    print(f'Saved {len(df)} rows -> {OUTPUT_CSV}')
    print()
    print(df['label'].map(LABEL_NAMES).value_counts().to_string())

## Step 10: Summary

In [ ]:
import matplotlib.pyplot as plt

df['label_name'] = df['label'].map(LABEL_NAMES)
counts = df['label_name'].value_counts()

plt.figure(figsize=(6, 4))
bars = plt.bar(counts.index, counts.values, color=['#2ecc71', '#e67e22', '#e74c3c'])
for b in bars:
    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 5,
             str(int(b.get_height())), ha='center', fontweight='bold')
plt.title('labeled_features.csv — Class Distribution')
plt.xlabel('Engagement State')
plt.ylabel('Sample Count')
plt.tight_layout()
plt.show()

print('Ready for training? (target: 200+ per class)')
for name, count in counts.items():
    mark = 'OK' if count >= 200 else f'needs {200 - count} more'
    print(f'  {name}: {count}  [{mark}]')
print()
print('Next: run MLP_training.ipynb with this CSV.')